# `wf_balance_inquiry` 단계별 Testbed

이 Notebook은 잔액조회 Workflow를 한 번에 실행하고 결과만 보는 용도가 아니다. 각 단계에서 다음 내용을 차례대로 확인한다.

1. Agent가 받은 사용자 발화
2. Agent 내부 Slot 추출 결과
3. Agent가 Backend Tool API에 보낸 요청과 Backend 응답
4. Agent가 Backend로 보낸 계좌 선택 UI Webhook
5. Backend가 검증하여 Agent에 전달한 Resume 값
6. 잔액조회 Tool API 요청과 결과 UI Webhook

기본 Scenario는 **계좌 후보가 여러 개여서 사용자 선택이 필요한 경우**다. Cell을 위에서부터 하나씩 실행한다.

## 전체 실행 흐름

```text
사용자 발화
  → [Agent 내부] 계좌 힌트 추출
  → [Agent → Backend API] 조회 가능 계좌 확인
  → [Agent → Backend Webhook] 계좌 선택 UI 요청
  → Workflow 중단
  → [Frontend → Backend] 사용자가 계좌 선택
  → [Backend → Agent] 검증된 account_ids로 Resume
  → [Agent → Backend API] 잔액 일괄 조회
  → [Agent → Backend Webhook] 잔액 결과 UI 전송
```

Agent는 계좌 소유권과 선택값을 직접 검증하지 않고, 금융 원장에도 접근하지 않는다.

## 0. 공통 설정과 출력 함수

In [ ]:
import json
import os

from pydantic import SecretStr

from agent.clients.backend import BackendClientConfig
from agent.testing import MockBackend
from agent.testing.balance_inquiry import (
    create_balance_backend_testbed,
    create_balance_mock_testbed,
)
from agent.workflow_contracts import WorkflowContractStore
from agent.workflows import extract_balance_slots_from_text
from agent.workflows.query_slot_extraction import extract_balance_slots_llm_first


def show_json(title, value):
    print(f"\n--- {title} ---")
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


CHAT_SESSION_ID = "chat_testbed_001"
EXECUTION_CONTEXT_ID = "exec_testbed_001"
THREAD_ID = "thread_balance_testbed_001"
INPUT_REQUEST_ID = "input_balance_testbed_001"

print("공통 설정 완료")

## 1. 관리시트 계약에서 실행 Step 확인

Notebook에서 임의로 Workflow 순서를 정하지 않는다. 관리시트에서 생성한 Manifest를 읽어 현재 Step, 통신 방식과 계약 ID를 확인한다.

In [ ]:
store = WorkflowContractStore()
workflow_contract = store.get_workflow("wf_balance_inquiry")

step_contracts = [
    {
        "order": step["step_order"],
        "step_id": step["step_id"],
        "mode": step["interaction_mode"],
        "contract_id": step.get("contract_id"),
        "next": step["route_summary"],
    }
    for step in workflow_contract["steps"]
]
show_json("Workflow Step 계약", step_contracts)

## 2. Step 1 — 사용자 발화에서 Slot 추출

### 입력

사용자가 계좌를 특정하지 않고 잔액을 요청한다. 이 Step은 Agent 내부에서 LLM 구조화 추출을 먼저 시도하고, 실패한 경우에만 결정적 규칙을 사용한다. Backend Tool API 호출은 없다.

기준 입력은 `잔액 보여줘`다. 다른 발화의 Slot 결과는 바로 다음 연습 Cell에서 별도로 비교한다.

In [ ]:
USER_MESSAGE = "잔액 보여줘"

slot_input = {"message": USER_MESSAGE}
slot_output = dict(await extract_balance_slots_llm_first(USER_MESSAGE))

show_json("Step 1 입력", slot_input)
show_json("Step 1 출력", slot_output)

assert slot_output == {
    "account_hint": None,
    "all_accounts_requested": False,
}

### 다른 발화를 직접 넣어보기

이 Cell은 Workflow 실행에 영향을 주지 않는 연습용이다.

In [ ]:
practice_messages = [
    "생활비 통장 잔액 보여줘",
    "신한은행 계좌 잔액 알려줘",
    "내 계좌 잔액을 전부 보여줘",
]

for message in practice_messages:
    show_json(message, dict(await extract_balance_slots_llm_first(message)))

show_json(
    "결정적 폴백 참고", dict(extract_balance_slots_from_text("생활비 통장 잔액 보여줘"))
)

## 3. Mock Backend 입력값 준비

Backend가 아직 없어도 실제 API와 같은 응답 Schema로 Workflow를 실행한다. 여기서는 계좌 후보 두 개를 반환하여 `selection_required` Route를 만든다.

아래 값은 Agent가 만든 값이 아니라 **Backend가 소유권과 조회 가능 상태를 검증한 뒤 반환했다고 가정한 값**이다.

In [ ]:
ACCOUNT_CANDIDATES = [
    {
        "account_id": "acc_001",
        "bank_name": "신한은행",
        "account_alias": "생활비 통장",
        "account_type": "checking",
        "masked_account_number": "110-***-123456",
        "currency": "KRW",
        "is_default": True,
        "status": "active",
    },
    {
        "account_id": "acc_002",
        "bank_name": "토스뱅크",
        "account_alias": "급여 통장",
        "account_type": "checking",
        "masked_account_number": "1000-***-654321",
        "currency": "KRW",
        "is_default": False,
        "status": "active",
    },
]

ACCOUNT_RESOLUTION_RESPONSE = {
    "account_resolution_outcome": "selection_required",
    "accounts": ACCOUNT_CANDIDATES,
    "account_ids": [],
}

show_json("Mock Backend 계좌 확인 응답 Data", ACCOUNT_RESOLUTION_RESPONSE)

### 선택 이후 사용할 Mock 잔액 결과

사용자가 두 번째 계좌를 선택한 뒤 Backend가 반환할 잔액 결과를 미리 등록한다.

In [ ]:
SELECTED_ACCOUNT_IDS = ["acc_002"]
BALANCE_RESULTS = [
    {
        "account_id": "acc_002",
        "bank_name": "토스뱅크",
        "account_alias": "급여 통장",
        "masked_account_number": "1000-***-654321",
        "balance": 3200000,
        "available_balance": 3180000,
        "currency": "KRW",
        "as_of": "2026-07-16T10:00:00+09:00",
    }
]

show_json("사용자가 선택할 계좌 ID", SELECTED_ACCOUNT_IDS)
show_json("Mock Backend 잔액조회 응답 Data", {"balance_results": BALANCE_RESULTS})

## 4. Mock Backend와 실제 Workflow 연결

예상하지 않은 Method나 Path가 호출되면 Mock Backend가 즉시 실패한다. 따라서 등록한 응답과 실제 Workflow 호출이 일치하는지도 함께 검증된다.

In [ ]:
mock_backend = MockBackend()
mock_backend.add_success(
    "GET",
    "/api/v1/agent-tools/accounts",
    ACCOUNT_RESOLUTION_RESPONSE,
)
mock_backend.add_success(
    "POST",
    "/api/v1/webhooks/agent",
    {"message_id": "message_input_001"},
)
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/accounts/balances:query",
    {"balance_results": BALANCE_RESULTS},
)
mock_backend.add_success(
    "POST",
    "/api/v1/webhooks/agent",
    {"message_id": "message_result_001"},
)

mock_config = BackendClientConfig(
    base_url="http://backend.test",
    agent_service_token=SecretStr("mock-service-token"),
    agent_webhook_secret=SecretStr("mock-webhook-secret"),
    retry_backoff_seconds=0,
)
testbed = create_balance_mock_testbed(
    mock_backend,
    mock_config,
    thread_id=THREAD_ID,
    input_request_id=INPUT_REQUEST_ID,
)

print("Mock Backend와 Workflow 연결 완료")

## 5. Step 2·3 실행 — 계좌 확인 후 사용자 입력 대기

`start()`를 호출하면 다음 중단 지점까지 실행한다.

- `extract_balance_slots`: Agent 내부 실행
- `resolve_balance_accounts`: Backend 계좌 확인 API 호출
- `request_balance_account_selection`: 선택 UI Webhook 생성 후 중단

즉, 결과는 완료가 아니라 `waiting`이어야 한다.

In [ ]:
start_request = {
    "message": USER_MESSAGE,
    "request_id": "req_balance_start_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
}
show_json("Workflow 시작 입력", start_request)

start_result = await testbed.start(**start_request)
show_json("Workflow 시작 결과", start_result.model_dump(mode="json"))

assert start_result.status == "waiting"
assert start_result.pending_interaction["input_request_id"] == INPUT_REQUEST_ID

### Step 2 실제 API 요청과 응답

Agent가 계좌를 직접 찾은 것이 아니다. 추출한 Slot을 `GET /accounts` Query로 보내고, Backend의 `selection_required` 결과를 그대로 Route에 사용한다.

In [ ]:
account_exchange = mock_backend.exchange_timeline(include_payload=True)[0]
show_json("Step 2 Agent → Backend 요청 / Backend → Agent 응답", account_exchange)

assert account_exchange["path"] == "/api/v1/agent-tools/accounts"
assert account_exchange["request"].get("account_hint") is None
assert (
    account_exchange["response"]["data"]["account_resolution_outcome"]
    == "selection_required"
)

### Step 3 실제 UI Webhook

계좌 후보 전체와 `input_request_id`가 `need_input` Webhook에 담긴다. Backend는 이 값을 저장하고 Frontend에 선택 화면을 전달한다.

In [ ]:
input_webhook_exchange = mock_backend.exchange_timeline(include_payload=True)[1]
input_webhook = input_webhook_exchange["request"]

show_json("Step 3 Agent → Backend need_input Webhook", input_webhook)
show_json("Step 3 Backend Webhook 수신 응답", input_webhook_exchange["response"])

assert input_webhook["event_type"] == "need_input"
assert input_webhook["metadata"]["input_request_id"] == INPUT_REQUEST_ID
assert len(input_webhook["metadata"]["ui"]["payload"]["accounts"]) == 2

### 중단 시점의 Agent State

아직 잔액은 조회하지 않았다. State에는 추출값, Backend가 검증한 계좌 후보와 입력 요청 ID만 들어 있다.

In [ ]:
waiting_state = await testbed.state(start_result.agent_thread_id)
show_json(
    "중단 시점 시스템 State",
    {
        "workflow_id": waiting_state.get("workflow_id"),
        "current_step_id": waiting_state.get("current_step_id"),
        "route_key": waiting_state.get("route_key"),
        "status": waiting_state.get("status"),
    },
)
show_json("중단 시점 Workflow Data", waiting_state["data"])
assert "balance_results" not in waiting_state["data"]

## 6. Step 4 — 사용자 선택과 Backend 검증 결과

실제 환경에서는 다음 순서로 진행된다.

1. Frontend가 사용자의 선택을 Backend로 전송한다.
2. Backend가 `input_request_id`, 사용자, 계좌 소유권과 조회 가능 상태를 검증한다.
3. Backend가 검증된 `account_ids`만 Agent Resume Endpoint로 전송한다.

Mock Testbed에서는 2번이 성공했다고 가정하여 `VALIDATED_RESUME_VALUE`를 직접 넣는다.

In [ ]:
FRONTEND_SELECTION = {
    "input_request_id": INPUT_REQUEST_ID,
    "account_ids": SELECTED_ACCOUNT_IDS,
}
VALIDATED_RESUME_VALUE = {
    "account_selection_outcome": "selected",
    "account_ids": SELECTED_ACCOUNT_IDS,
}

show_json("Frontend → Backend 사용자 선택 예시", FRONTEND_SELECTION)
show_json("Backend 검증 후 Agent에 전달할 Resume Value", VALIDATED_RESUME_VALUE)

## 7. Step 5·6 실행 — Resume, 잔액조회와 결과 전송

검증된 Resume 값으로 Workflow를 재개한다. Agent는 선택값을 다시 계좌 확인 API에 보내지 않고 바로 잔액 일괄조회 API를 호출한다.

In [ ]:
resume_request = {
    "agent_thread_id": start_result.agent_thread_id,
    "request_id": "req_balance_resume_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "input_request_id": INPUT_REQUEST_ID,
    "value": VALIDATED_RESUME_VALUE,
}
show_json("Backend → Agent Resume 요청", resume_request)

final_result = await testbed.resume_input(**resume_request)
show_json("Resume 이후 Workflow 결과", final_result.model_dump(mode="json"))

assert final_result.status == "completed"

### Step 5 실제 잔액조회 API 요청과 응답

요청에는 Backend가 검증한 `account_ids` 배열만 들어간다.

In [ ]:
balance_exchange = next(
    exchange
    for exchange in mock_backend.exchange_timeline(include_payload=True)
    if exchange["path"] == "/api/v1/agent-tools/accounts/balances:query"
)
show_json("Step 5 Agent → Backend 요청 / Backend → Agent 응답", balance_exchange)

assert balance_exchange["request"] == {"account_ids": SELECTED_ACCOUNT_IDS}

### Step 6 실제 결과 UI Webhook

Backend가 반환한 잔액 결과를 Frontend가 렌더링할 `balance_result` UI Payload로 변환하여 전송한다.

In [ ]:
result_webhook_exchange = mock_backend.exchange_timeline(include_payload=True)[-1]
result_webhook = result_webhook_exchange["request"]

show_json("Step 6 Agent → Backend 결과 Webhook", result_webhook)
show_json("Frontend가 표시할 UI Payload", result_webhook["metadata"]["ui"])

assert result_webhook["event_type"] == "component"
assert result_webhook["metadata"]["ui"]["type"] == "balance_result"

### 완료 시점의 Agent State

선택된 계좌와 Backend 잔액 결과가 반영되고 `input_request_id`는 제거되어야 한다.

In [ ]:
completed_state = await testbed.state(start_result.agent_thread_id)
show_json(
    "완료 시점 시스템 State",
    {
        "current_step_id": completed_state.get("current_step_id"),
        "route_key": completed_state.get("route_key"),
        "status": completed_state.get("status"),
    },
)
show_json("완료 시점 Workflow Data", completed_state["data"])
assert completed_state["data"]["account_ids"] == SELECTED_ACCOUNT_IDS
assert completed_state["data"]["input_request_id"] is None

## 8. 전체 통신 순서와 자동 확인

마지막으로 Workflow가 실제로 호출한 모든 HTTP 요청을 순서대로 확인한다. 인증 Token과 Secret Header는 출력하지 않는다.

In [ ]:
timeline = testbed.request_timeline()
show_json("Agent HTTP 호출 순서", timeline)

paths = [item["path"] for item in timeline]
checks = {
    "계좌 확인 API는 1회만 호출": paths.count("/api/v1/agent-tools/accounts") == 1,
    "need_input Webhook 전송": timeline[1].get("event_type") == "need_input",
    "Resume 후 잔액조회 API 호출": "/api/v1/agent-tools/accounts/balances:query"
    in paths,
    "결과 component Webhook 전송": timeline[-1].get("event_type") == "component",
    "Workflow 완료": final_result.status == "completed",
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(checks.values())
mock_backend.assert_all_responses_used()

## 9. 다른 Route는 무엇이 달라지는가

| Backend 계좌 확인 결과 | 다음 동작 |
| --- | --- |
| `resolved` | 선택 UI 없이 바로 잔액조회 API 호출 |
| `selection_required` | 현재 Notebook처럼 선택 UI와 Resume 수행 |
| `no_accounts` | 빈 계좌 UI를 전송하고 종료 |
| 계약 오류 | 공통 오류 UI를 전송하고 종료 |

이 Route들은 `agent/tests/test_balance_reference_workflow.py`에서도 자동 검증한다. Notebook은 가장 많은 경계를 확인할 수 있는 `selection_required`를 단계별 기준 Scenario로 사용한다.

## 10. 실제 Backend로 전환하기

아래 Cell은 기본적으로 실행하지 않는다. 실제 Backend 테스트 Context가 준비되면 환경변수를 설정하고 `RUN_REAL_BACKEND = True`로 변경한다.

이 단계는 Agent Tool API와 Webhook까지 검증한다. Frontend 입력과 Backend 검증을 포함한 전체 E2E는 Backend 실행 시작·Resume Endpoint와 브라우저 테스트에서 확인한다.

In [ ]:
RUN_REAL_BACKEND = False

if RUN_REAL_BACKEND:
    required_variables = [
        "BACKEND_BASE_URL",
        "AGENT_SERVICE_TOKEN",
        "AGENT_WEBHOOK_SECRET",
        "TESTBED_CHAT_SESSION_ID",
        "TESTBED_EXECUTION_CONTEXT_ID",
    ]
    missing = [name for name in required_variables if not os.getenv(name)]
    if missing:
        raise RuntimeError(f"Backend Mode 환경변수가 없습니다: {missing}")

    real_config = BackendClientConfig(
        base_url=os.environ["BACKEND_BASE_URL"],
        agent_service_token=SecretStr(os.environ["AGENT_SERVICE_TOKEN"]),
        agent_webhook_secret=SecretStr(os.environ["AGENT_WEBHOOK_SECRET"]),
    )
    real_testbed = create_balance_backend_testbed(real_config)
    real_result = await real_testbed.start(
        message=USER_MESSAGE,
        request_id="req_real_balance_001",
        chat_session_id=os.environ["TESTBED_CHAT_SESSION_ID"],
        execution_context_id=os.environ["TESTBED_EXECUTION_CONTEXT_ID"],
    )
    show_json("실제 Backend 실행 결과", real_result.model_dump(mode="json"))
    show_json("실제 Backend 요청 요약", real_testbed.request_timeline())
    await real_testbed.aclose()
else:
    print("실제 Backend 호출을 건너뜁니다. Mock 단계별 Testbed만 실행했습니다.")

## 11. Testbed 종료

In [ ]:
await testbed.aclose()
print("잔액조회 단계별 Testbed 종료 완료")